# 第15篇｜多表合并：merge / join / concat 实战指南

> 这是「数据分析从入门到精通」系列的第 15 篇。现实工作里，数据往往分散在多张表里——用户表、订单表、商品表……这篇把 merge / join / concat 三种合表方式讲透，以后遇到多表关联，不慌。

---

嗨，我是小荷～

今天这篇，我想先聊一个真实的困境。

你在公司拿到三张表：**订单表**（记录每笔交易）、**用户表**（记录用户信息）、**商品表**（记录商品详情）。老板让你分析"哪类用户最喜欢买什么品类的商品"。

数据就在那里，但分散在三张表里，怎么把它们拼在一起？

这让我想到萧何管粮草的故事——各路诸侯送来的粮草账目格式不一样，有的按县记录，有的按将领记录。萧何整合这些信息的能力，才是让大汉后勤稳如磐石的关键。Pandas 的多表合并，就是数据版的"萧何整合术"。

---

## 🔑 三种合并方式，先搞清楚区别

| 方法 | 类比 | 适用场景 |
|------|------|---------|
| `pd.merge()` | SQL 的 JOIN | 按某列关联两张表（最常用） |
| `df.join()` | 按索引关联 | 两表索引相同时快捷合并 |
| `pd.concat()` | 表格拼接 | 纵向堆叠 / 横向并排 |

一句话记法：**merge 按列关联，join 按索引，concat 直接拼**。

---

## 一、pd.merge()：按列关联

### 基础语法

先来看最常用的语法格式：


In [ ]:
import pandas as pd

result = pd.merge(left, right, on='关联列', how='join类型')


`how` 参数对应 SQL 的四种 JOIN：

| how 值 | 类比 SQL | 保留哪些行 |
|--------|---------|---------|
| `inner` | INNER JOIN | 只保留两表都有的行（默认） |
| `left` | LEFT JOIN | 保留左表全部 |
| `right` | RIGHT JOIN | 保留右表全部 |
| `outer` | FULL OUTER JOIN | 两表全部保留，缺失用 NaN |

### 实战演示

看个具体例子：


In [1]:
import pandas as pd

# 模拟三张表
orders = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1005],
    'user_id':  [201, 202, 201, 203, 204],
    'product_id': ['P01', 'P02', 'P01', 'P03', 'P02'],
    'amount':   [299, 599, 299, 199, 599]
})

users = pd.DataFrame({
    'user_id': [201, 202, 203, 205],  # 注意：204 不在用户表里，205 没有订单
    'name':    ['张三', '李四', '王五', '赵六'],
    'city':    ['北京', '上海', '广州', '深圳']
})

products = pd.DataFrame({
    'product_id': ['P01', 'P02', 'P03'],
    'category':   ['数码', '服装', '食品'],
    'brand':      ['品牌A', '品牌B', '品牌C']
})

# 第一步：订单表关联用户表
order_user = pd.merge(orders, users, on='user_id', how='left')
print(order_user)


   order_id  user_id product_id  amount name city
0      1001      201        P01     299   张三   北京
1      1002      202        P02     599   李四   上海
2      1003      201        P01     299   张三   北京
3      1004      203        P03     199   王五   广州
4      1005      204        P02     599  NaN  NaN


输出：


接下来再关联商品表，把订单、用户、商品三张表拼到一起：


In [2]:
# 第二步：再关联商品表
full_data = pd.merge(order_user, products, on='product_id', how='left')
print(full_data)


   order_id  user_id product_id  amount name city category brand
0      1001      201        P01     299   张三   北京       数码   品牌A
1      1002      202        P02     599   李四   上海       服装   品牌B
2      1003      201        P01     299   张三   北京       数码   品牌A
3      1004      203        P03     199   王五   广州       食品   品牌C
4      1005      204        P02     599  NaN  NaN       服装   品牌B


输出：


In [ ]:
   order_id  user_id product_id  amount name  city category brand
0      1001      201        P01     299   张三    北京       数码  品牌A
1      1002      202        P02     599   李四    上海       服装  品牌B
2      1003      201        P01     299   张三    北京       数码  品牌A
3      1004      203        P03     199   王五    广州       食品  品牌C
4      1005      204        P02     599  NaN   NaN       服装  品牌B


三张表合并完成！现在每行订单都带着用户信息和商品信息了。

---

### left / right / inner / outer 的区别

来看看这几种方式有什么不同：


In [3]:
# 演示四种 how 的区别
df_a = pd.DataFrame({'id': [1, 2, 3], 'val_a': ['A1', 'A2', 'A3']})
df_b = pd.DataFrame({'id': [2, 3, 4], 'val_b': ['B2', 'B3', 'B4']})

# inner：只保留两边都有的
print("inner：")
print(pd.merge(df_a, df_b, on='id', how='inner'))
# 结果：id=2, id=3

# left：以左表为准
print("\nleft：")
print(pd.merge(df_a, df_b, on='id', how='left'))
# 结果：id=1,2,3；id=1 的 val_b 是 NaN

# outer：全保留
print("\nouter：")
print(pd.merge(df_a, df_b, on='id', how='outer'))
# 结果：id=1,2,3,4；缺失的填 NaN


inner：
   id val_a val_b
0   2    A2    B2
1   3    A3    B3

left：
   id val_a val_b
0   1    A1   NaN
1   2    A2    B2
2   3    A3    B3

outer：
   id val_a val_b
0   1    A1   NaN
1   2    A2    B2
2   3    A3    B3
3   4   NaN    B4


> 💡 **工作中 90% 的场景用 left join**，以主表为准，关联补充信息，不丢数据。

---

### 关联列名不同时怎么办？

来关联一下：


In [5]:
# 左表叫 user_id，右表叫 uid
orders2 = pd.DataFrame({'order_id': [1, 2], 'user_id': [101, 102]})
users2  = pd.DataFrame({'uid': [101, 102], 'name': ['张三', '李四']})

result = pd.merge(orders2, users2, left_on='user_id', right_on='uid')
print(result)


   order_id  user_id  uid name
0         1      101  101   张三
1         2      102  102   李四


`left_on` 和 `right_on` 分别指定两表的关联列，即使名字不同也能关联。

---

### 多列联合关联

来关联一下：


In [6]:
# 同时按年份和城市关联
df1 = pd.DataFrame({'year': [2023, 2023, 2024], 'city': ['北京', '上海', '北京'], 'sales': [100, 200, 150]})
df2 = pd.DataFrame({'year': [2023, 2024], 'city': ['北京', '北京'], 'target': [120, 160]})

result = pd.merge(df1, df2, on=['year', 'city'], how='left')
print(result)


   year city  sales  target
0  2023   北京    100   120.0
1  2023   上海    200     NaN
2  2024   北京    150   160.0


---

## 二、pd.concat()：拼接表格

### 纵向拼接（堆叠行）

来看看怎么拼接：


In [9]:
# 三个月的销售数据，分三张表
jan = pd.DataFrame({'month': ['2024-01']*3, 'product': ['A','B','C'], 'sales': [100, 200, 150]})
feb = pd.DataFrame({'month': ['2024-02']*3, 'product': ['A','B','C'], 'sales': [120, 180, 160]})
mar = pd.DataFrame({'month': ['2024-03']*3, 'product': ['A','B','C'], 'sales': [110, 220, 140]})

# 一次性拼起来
all_months = pd.concat([jan, feb, mar], ignore_index=True)
print(all_months)


     month product  sales
0  2024-01       A    100
1  2024-01       B    200
2  2024-01       C    150
3  2024-02       A    120
4  2024-02       B    180
5  2024-02       C    160
6  2024-03       A    110
7  2024-03       B    220
8  2024-03       C    140


`ignore_index=True` 让新表的索引从 0 重新编，不然会保留原来的 0,1,2,0,1,2...看起来很混乱。

---

### 横向拼接（并排列）

来看看怎么拼接：


In [10]:
# 两个表的列数相同，索引一致，横向并排
df_left  = pd.DataFrame({'name': ['张三', '李四', '王五']}, index=[0, 1, 2])
df_right = pd.DataFrame({'score': [90, 85, 92]}, index=[0, 1, 2])

result = pd.concat([df_left, df_right], axis=1)
print(result)


  name  score
0   张三     90
1   李四     85
2   王五     92


`axis=1` 代表横向拼接，`axis=0`（默认）是纵向。

---

### concat 常见陷阱：列名不一致

有些坑踩过才知道，提前看看：


In [11]:
# 注意：列名不一致会产生很多 NaN
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df2 = pd.DataFrame({'B': [5, 6], 'C': [7, 8]})  # 没有 A，有 C

result = pd.concat([df1, df2], ignore_index=True)
print(result)
#      A  B    C
# 0  1.0  3  NaN
# 1  2.0  4  NaN
# 2  NaN  5  7.0
# 3  NaN  6  8.0


     A  B    C
0  1.0  3  NaN
1  2.0  4  NaN
2  NaN  5  7.0
3  NaN  6  8.0


两表不共同拥有的列会自动补 NaN，要注意。

---

## 三、df.join()：按索引合并

`join` 是 `merge` 的简化版，默认按**索引**关联：


In [12]:
df_a = pd.DataFrame({'val_a': [10, 20, 30]}, index=['x', 'y', 'z'])
df_b = pd.DataFrame({'val_b': [100, 200, 300]}, index=['x', 'y', 'z'])

result = df_a.join(df_b)
print(result)


   val_a  val_b
x     10    100
y     20    200
z     30    300


适合两表索引相同时快速合并，但日常数据大多按列 ID 关联，所以 `merge` 用得更多。

---

## 四、🔧 综合实战：三表关联分析用户消费

学了一堆理论，来个完整的实战练练手——把前面学的知识点串起来：


In [13]:
import pandas as pd
import numpy as np

# ── 构造数据 ──
np.random.seed(42)
n = 100

orders = pd.DataFrame({
    'order_id':    range(1001, 1001+n),
    'user_id':     np.random.choice(range(201, 221), n),       # 20个用户
    'product_id':  np.random.choice(['P01','P02','P03','P04','P05'], n),
    'amount':      np.random.randint(50, 500, n),
    'order_date':  pd.date_range('2024-01-01', periods=n, freq='3D')
})

users = pd.DataFrame({
    'user_id': range(201, 221),
    'name':    [f'用户{i}' for i in range(1, 21)],
    'city':    np.random.choice(['北京', '上海', '广州', '深圳', '杭州'], 20),
    'level':   np.random.choice(['普通', '银卡', '金卡', '钻石'], 20)
})

products = pd.DataFrame({
    'product_id': ['P01','P02','P03','P04','P05'],
    'category':   ['数码', '服装', '食品', '美妆', '图书'],
    'price_tier': ['高', '中', '低', '中', '低']
})

# ── 三表合并 ──
full = pd.merge(orders, users, on='user_id', how='left')
full = pd.merge(full, products, on='product_id', how='left')

print("合并后数据形状：", full.shape)
print(full.head(3))

# ── 分析1：各城市消费总额 ──
city_sales = full.groupby('city')['amount'].agg(['sum', 'count', 'mean']).round(1)
city_sales.columns = ['总金额', '订单数', '平均客单价']
city_sales = city_sales.sort_values('总金额', ascending=False)
print("\n各城市消费：")
print(city_sales)

# ── 分析2：各品类不同会员等级的平均消费 ──
pivot = full.pivot_table(
    values='amount',
    index='category',
    columns='level',
    aggfunc='mean'
).round(1)
print("\n各品类 × 会员等级 平均消费：")
print(pivot)

# ── 分析3：哪类用户最喜欢哪个品类（订单数） ──
cross = full.groupby(['level', 'category'])['order_id'].count().unstack(fill_value=0)
print("\n会员等级 × 品类 订单数分布：")
print(cross)


合并后数据形状： (100, 10)
   order_id  user_id product_id  amount order_date  name city level category  \
0      1001      207        P03     375 2024-01-01   用户7   北京    普通       食品   
1      1002      220        P04     398 2024-01-04  用户20   杭州    普通       美妆   
2      1003      215        P03     308 2024-01-07  用户15   广州    钻石       食品   

  price_tier  
0          低  
1          中  
2          低  

各城市消费：
       总金额  订单数  平均客单价
city                  
广州    8182   28  292.2
深圳    7836   26  301.4
北京    6352   21  302.5
杭州    6338   22  288.1
上海     643    3  214.3

各品类 × 会员等级 平均消费：
level        普通     金卡     钻石     银卡
category                            
图书        315.8  116.6  437.0  252.7
数码        249.1  305.1  245.2  331.2
服装        303.7  433.0  318.0  197.0
美妆        315.5  346.0  249.7  441.0
食品        321.0  284.5  295.0  326.8

会员等级 × 品类 订单数分布：
category  图书  数码  服装  美妆  食品
level                       
普通         5   9   7  12  12
金卡         5   7   1   3   2
钻石         3   5   4

萧何当年整合各路粮草账目，靠的是系统化的记录和关联能力。我们的三表合并也是一样：**把分散的数据整合成一张全面的"大宽表"，分析才能真正展开**。

---

## 五、📋 合并方式速查表

| 需求 | 推荐方法 | 关键参数 |
|------|---------|---------|
| 按某列关联两张表 | `pd.merge()` | `on`, `how` |
| 关联列名不同 | `pd.merge()` | `left_on`, `right_on` |
| 多列联合关联 | `pd.merge()` | `on=['col1','col2']` |
| 纵向堆叠多张表 | `pd.concat()` | `axis=0`, `ignore_index=True` |
| 横向并排多张表 | `pd.concat()` | `axis=1` |
| 按索引快速合并 | `df.join()` | `how` |

---

## 六、⚠️ 常见坑

1. **关联后行数莫名变多**：说明关联列有重复值（一对多关联），排查数据是否有重复键。
2. **大量 NaN 出现**：检查两表的关联列值是否匹配，注意大小写、空格、数据类型。
3. **concat 后索引混乱**：纵向拼接一定要加 `ignore_index=True`。
4. **类型不一致导致关联失败**：一张表的 `user_id` 是 int，另一张是 str，必须先统一类型。


In [ ]:
# 类型不一致的处理方式
df['user_id'] = df['user_id'].astype(str)


---

## 七、📝 小结

| 知识点 | 要点 |
|--------|------|
| merge how 参数 | inner/left/right/outer，工作中最常用 left |
| 列名不同时合并 | left_on + right_on |
| 多表连续合并 | 链式 merge，每次合并一张 |
| 纵向堆叠 | pd.concat + ignore_index=True |
| 横向拼接 | pd.concat + axis=1 |

---

## 八、🏋️ 课后练习

1. 用 `inner join` 关联订单表和用户表，看看哪些 user_id 在用户表里找不到匹配？
2. 把 1 月、2 月、3 月三张销售表纵向合并成一张，并验证总行数是否正确。
3. 尝试用 `outer join` 合并两张表，并找出哪些行出现了 NaN，分析原因。

In [18]:
# ========== 任务1：inner join，找出用户表中找不到匹配的user_id ==========
print("\n" + "=" * 50)
print("任务1：inner join 关联订单表和用户表")
print("=" * 50)
merged_inner = pd.merge(orders, users, on='user_id', how='inner')
print("inner join 结果:")
print(merged_inner)

# 找出在用户表中找不到匹配的user_id
unmatched = set(orders['user_id']) - set(users['user_id'])
print(f"\n在用户表中找不到匹配的 user_id: {sorted(unmatched)}")

# 用 left join 更直观地看到
merged_left = pd.merge(orders, users, on='user_id', how='left', indicator=True)
print("\nleft join 标记（indicator）:")
print(merged_left[['order_id', 'user_id', '_merge']])

# ========== 任务2：纵向合并1月、2月、3月销售表 ==========
print("\n" + "=" * 50)
print("任务2：纵向合并三个月销售表")
print("=" * 50)
# 先确保 order_date 是 datetime 类型
orders['order_date'] = pd.to_datetime(orders['order_date'])

# 分别取出 1月、2月、3月
jan = orders[orders['order_date'].dt.month == 1]
feb = orders[orders['order_date'].dt.month == 2]
mar = orders[orders['order_date'].dt.month == 3]

# 拼成一张表
q1 = pd.concat([jan, feb, mar], ignore_index=True)
print(f"1月: {len(jan)} 行, 2月: {len(feb)} 行, 3月: {len(mar)} 行")
print(f"合并后总行数: {len(q1)} 行")
print(f"验证: {len(jan)} + {len(feb)} + {len(mar)} = {len(jan) + len(feb) + len(mar)}")
print(q1)

# ========== 任务3：outer join 合并，找出NaN行 ==========
print("\n" + "=" * 50)
print("任务3：outer join 合并并找出NaN行")
print("=" * 50)
# 两张表用于 outer join
table_a = pd.DataFrame({
    'key': ['A', 'B', 'C', 'D'],
    'value_a': [10, 20, 30, 40]
})
table_b = pd.DataFrame({
    'key': ['A', 'B', 'E', 'F'],
    'value_b': [100, 200, 300, 400]
})
merged_outer = pd.merge(table_a, table_b, on='key', how='outer', indicator=True)
print("table_a:")
print(table_a)
print("\ntable_b:")
print(table_b)
print("\nouter join 结果:")
print(merged_outer)

# 找出包含NaN的行
nan_rows = merged_outer[merged_outer.isnull().any(axis=1)]
print(f"\n包含 NaN 的行:")
print(nan_rows)


任务1：inner join 关联订单表和用户表
inner join 结果:
    order_id  user_id product_id  amount order_date  name city level
0       1001      207        P03     375 2024-01-01   用户7   北京    普通
1       1002      220        P04     398 2024-01-04  用户20   杭州    普通
2       1003      215        P03     308 2024-01-07  用户15   广州    钻石
3       1004      211        P03     197 2024-01-10  用户11   深圳    银卡
4       1005      208        P01     301 2024-01-13   用户8   深圳    钻石
..       ...      ...        ...     ...        ...   ...  ...   ...
95      1096      212        P03     424 2024-10-12  用户12   杭州    普通
96      1097      202        P01      71 2024-10-15   用户2   广州    普通
97      1098      201        P01     287 2024-10-18   用户1   杭州    银卡
98      1099      216        P04     207 2024-10-21  用户16   北京    普通
99      1100      205        P03      87 2024-10-24   用户5   上海    钻石

[100 rows x 8 columns]

在用户表中找不到匹配的 user_id: []

left join 标记（indicator）:
    order_id  user_id _merge
0       1001      207   bot

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 16 篇：数据透视表与交叉分析**
>
> 多表合并之后，数据已经整合好了。但一份多维的大宽表，怎么快速看出各维度的对比？下篇带你用数据透视表做多维交叉分析，一行代码让老板看了直点头。

---

*跟着小荷，数据分析路上不迷路～*
*（萧何管粮草靠整合，你分析数据靠 merge 😄）*